# Advanced RAG-LLM Action Planning with Vector Embeddings

This notebook creates a RAG knowledge base from JSON files using vector embeddings and generates action plans using LLM.

**Features:**
- Proper JSON loading and processing
- Text chunking with overlap
- Vector embeddings (OpenAI)
- Vector similarity search (FAISS)
- Semantic search instead of keyword matching

In [ ]:
# 0. Install Required Packages (run once)
# Uncomment and run this cell if you need to install dependencies
"""
%pip install langchain openai faiss-cpu python-dotenv
# For GPU support (optional): %pip install faiss-gpu
"""

In [ ]:
# 1. Setup and Load Predictions
import os
import json
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# Paths
rag_knowledge_dir = Path("RAG_docs/knowledge")
rag_predictions_dir = Path("RAG_docs/predictions")
results_action_dir = Path("RAG_docs/action_plans")
os.makedirs(results_action_dir, exist_ok=True)

# Load prediction file (find latest)
prediction_files = list(rag_predictions_dir.glob("sample_predictions_detailed_*.json"))
if prediction_files:
    latest_prediction = max(prediction_files, key=lambda p: p.stat().st_mtime)
    with open(latest_prediction, "r", encoding="utf-8") as f:
        predictions_data = json.load(f)
    print(f"Loaded {len(predictions_data)} predictions from {latest_prediction.name}")
else:
    print("No prediction files found!")
    predictions_data = []


In [ ]:
# 2. Build RAG Knowledge Base from JSON Files with Proper Processing
import json
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.schema import Document

# Initialize text splitter for chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # Characters per chunk
    chunk_overlap=200,  # Overlap between chunks for context preservation
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]  # Split on these in order
)

rag_documents = []
rag_metadata = []

# Load all JSON files from knowledge folder
if rag_knowledge_dir.exists():
    json_files = list(rag_knowledge_dir.glob("*.json"))
    print(f"Found {len(json_files)} JSON files in knowledge folder")
    
    for json_file in json_files:
        print(f"  Loading: {json_file.name}")
        try:
            with open(json_file, "r", encoding="utf-8") as f:
                data = json.load(f)
            
            # Process JSON data - handle array of objects
            if isinstance(data, list):
                for idx, item in enumerate(data):
                    # Extract key fields for RAG content
                    # For DDoS dataset: title + text
                    if isinstance(item, dict):
                        # Combine title and text as main content
                        title = item.get("title", "")
                        text = item.get("text", "")
                        
                        # Create formatted document text
                        doc_text = f"Title: {title}\n\nContent: {text}"
                        
                        # Create metadata
                        metadata = {
                            "file": json_file.name,
                            "source": json_file.stem,
                            "item_index": idx,
                            "title": title[:100] if title else "Untitled"  # Truncate for metadata
                        }
                        
                        # Split into chunks if text is long
                        chunks = text_splitter.split_text(doc_text)
                        
                        for chunk_idx, chunk in enumerate(chunks):
                            rag_documents.append(Document(
                                page_content=chunk,
                                metadata={**metadata, "chunk_index": chunk_idx}
                            ))
                            
            elif isinstance(data, dict):
                # Handle single dict - flatten key-value pairs
                doc_text = ""
                for key, value in data.items():
                    if isinstance(value, (str, int, float, bool)):
                        doc_text += f"{key}: {value}\n"
                    elif isinstance(value, (list, dict)):
                        doc_text += f"{key}: {json.dumps(value, ensure_ascii=False)}\n"
                
                metadata = {
                    "file": json_file.name,
                    "source": json_file.stem,
                    "type": "dict"
                }
                
                # Split into chunks
                chunks = text_splitter.split_text(doc_text)
                for chunk_idx, chunk in enumerate(chunks):
                    rag_documents.append(Document(
                        page_content=chunk,
                        metadata={**metadata, "chunk_index": chunk_idx}
                    ))
                    
        except Exception as e:
            print(f"    Error loading {json_file.name}: {e}")
            import traceback
            traceback.print_exc()

print(f"\nTotal RAG document chunks created: {len(rag_documents)}")


In [ ]:
# 3a. Optional: Load Existing Vector Store (if previously saved)
# Uncomment and use this if you've already created a vector store and want to reload it
"""
vector_store_path = rag_knowledge_dir.parent / "vector_store"
if vector_store_path.exists() and vector_store is None:
    try:
        print(f"Loading existing vector store from {vector_store_path}...")
        embeddings = OpenAIEmbeddings(openai_api_key=os.getenv("OPENAI_API_KEY"))
        vector_store = FAISS.load_local(str(vector_store_path), embeddings)
        print("✓ Vector store loaded successfully!")
    except Exception as e:
        print(f"Could not load vector store: {e}")
        print("Will create a new one in the next cell.")
"""

In [ ]:
# 3. Create Vector Store with Embeddings
import os

# Check for OpenAI API key
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    print("WARNING: OPENAI_API_KEY not set. Vector search will not work.")
    vector_store = None
else:
    try:
        # Initialize embeddings
        embeddings = OpenAIEmbeddings(openai_api_key=api_key)
        
        # Create vector store from documents
        print("Creating vector embeddings and FAISS index...")
        print(f"Processing {len(rag_documents)} document chunks...")
        
        vector_store = FAISS.from_documents(rag_documents, embeddings)
        
        print(f"✓ Vector store created successfully!")
        print(f"  - Total chunks: {len(rag_documents)}")
        print(f"  - Embedding dimension: 1536 (OpenAI text-embedding-ada-002)")
        
        # Optional: Save vector store to disk for reuse
        vector_store_path = rag_knowledge_dir.parent / "vector_store"
        vector_store.save_local(str(vector_store_path))
        print(f"  - Saved to: {vector_store_path}")
        
    except Exception as e:
        print(f"ERROR creating vector store: {e}")
        import traceback
        traceback.print_exc()
        vector_store = None

# 4. Vector-based RAG Search Function
def search_rag_knowledge(query, top_k=3, score_threshold=0.7):
    """
    Semantic search using vector embeddings.
    
    Args:
        query: Search query string
        top_k: Number of results to return
        score_threshold: Minimum similarity score (0-1)
    
    Returns:
        List of relevant documents with metadata
    """
    if vector_store is None:
        print("WARNING: Vector store not available. Returning empty results.")
        return []
    
    try:
        # Perform similarity search
        results = vector_store.similarity_search_with_score(query, k=top_k)
        
        # Format results
        formatted_results = []
        for doc, score in results:
            # Convert similarity distance to similarity score (FAISS uses L2 distance)
            # Lower distance = higher similarity
            # Normalize to 0-1 range (approximate)
            similarity_score = 1 / (1 + score) if score > 0 else 1.0
            
            if similarity_score >= score_threshold:
                formatted_results.append({
                    "content": doc.page_content,
                    "metadata": doc.metadata,
                    "score": similarity_score,
                    "distance": float(score)
                })
        
        return formatted_results
        
    except Exception as e:
        print(f"ERROR in vector search: {e}")
        import traceback
        traceback.print_exc()
        return []

print("\n✓ Vector-based RAG search function ready")


In [ ]:
# 5. Create Enhanced Prompt with Vector RAG Results
def create_simple_prompt(sample_data, top_k=3):
    """Create prompt from prediction details and vector-based RAG search."""
    # Get prediction details
    sample_id = sample_data.get("sample_id", 0)
    attack_type = sample_data.get("predicted_label", "UNKNOWN")
    confidence = sample_data.get("confidence", 0.0)
    
    # Get party contributions if available
    shap_explanation = sample_data.get("shap_explanation", {})
    party_contributions = shap_explanation.get("party_contributions_pct", {})
    
    # Search RAG knowledge base with semantic search
    query = f"{attack_type} attack mitigation prevention strategies"
    rag_results = search_rag_knowledge(query, top_k=top_k, score_threshold=0.6)
    
    # Build RAG context from vector search results
    rag_context = ""
    if rag_results:
        rag_context = "\n\nRAG Knowledge Base Context (from vector similarity search):\n"
        for idx, result in enumerate(rag_results, 1):
            metadata = result.get("metadata", {})
            title = metadata.get("title", "Unknown")
            file_name = metadata.get("file", "unknown")
            similarity = result.get("score", 0.0)
            
            rag_context += f"\n[{idx}] {title} (Similarity: {similarity:.2%})\n"
            rag_context += f"Source: {file_name}\n"
            rag_context += f"Content: {result['content'][:800]}...\n"  # First 800 chars
    else:
        rag_context = "\n\nRAG Knowledge Base: No relevant documents found above similarity threshold."
    
    # Build party contributions text
    party_text = ""
    if party_contributions:
        party_text = "\nParty Contributions:\n"
        for party, contrib in sorted(party_contributions.items(), key=lambda x: x[1], reverse=True)[:3]:
            party_text += f"  - {party}: {contrib:.1%}\n"
    
    # Create enhanced prompt
    prompt = f"""Analyze this attack detection and provide a comprehensive action plan based on the knowledge base context.

PREDICTION DETAILS:
- Sample ID: {sample_id}
- Attack Type: {attack_type}
- Confidence: {confidence:.1%}
{party_text}
{rag_context}

Based on the above information, provide a detailed action plan in JSON format:
{{
  "threat_level": "Critical|High|Medium|Low",
  "primary_actions": ["Action 1", "Action 2", ...],
  "supporting_actions": ["Action 1", "Action 2", ...],
  "reasoning": "Brief explanation referencing the knowledge base",
  "execution_priority": "Immediate|Fast-track|Standard|Monitor",
  "knowledge_sources_used": ["List of titles from RAG results"]
}}"""
    
    return prompt, rag_results

print("✓ Enhanced prompt creation function ready")


In [ ]:
# 6. Call LLM API
def call_llm_api(prompt):
    """Call LLM API with enhanced prompt."""
    try:
        from openai import OpenAI
    except ImportError:
        print("ERROR: openai package not installed. Run: pip install openai")
        return None
    
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        print("ERROR: OPENAI_API_KEY not set in .env file")
        return None
    
    model = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
    temperature = float(os.getenv("OPENAI_TEMPERATURE", "0.3"))
    
    client = OpenAI(api_key=api_key)
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are a cybersecurity expert specializing in DDoS attack mitigation. Analyze the provided context and return only valid JSON."},
            {"role": "user", "content": prompt}
        ],
        temperature=temperature
    )
    
    # Parse response
    response_text = response.choices[0].message.content.strip()
    try:
        # Try to extract JSON
        start = response_text.find('{')
        end = response_text.rfind('}') + 1
        if start >= 0 and end > start:
            return json.loads(response_text[start:end])
    except Exception as e:
        print(f"Warning: Could not parse JSON response: {e}")
    
    # Fallback
    return {
        "threat_level": "Unknown",
        "primary_actions": ["Unable to parse LLM response"],
        "supporting_actions": [],
        "reasoning": response_text[:200],
        "execution_priority": "Standard",
        "knowledge_sources_used": []
    }

# 7. Process Sample with Vector RAG
# Note: Currently processes first sample only. To process all, change to: for sample in predictions_data[:10]
if len(predictions_data) > 0:
    sample = predictions_data[0]  # Process first sample
    
    print("="*80)
    print("PROCESSING SAMPLE WITH VECTOR RAG")
    print("="*80)
    print(f"Sample ID: {sample.get('sample_id')}")
    print(f"Attack Type: {sample.get('predicted_label')}")
    print(f"Confidence: {sample.get('confidence', 0):.1%}")
    
    # Create prompt with vector-based RAG search
    prompt, rag_results = create_simple_prompt(sample, top_k=5)
    
    print(f"\n✓ Found {len(rag_results)} relevant RAG documents (vector similarity search)")
    if rag_results:
        print("\nTop RAG Results:")
        for idx, result in enumerate(rag_results[:3], 1):
            metadata = result.get("metadata", {})
            title = metadata.get("title", "Unknown")
            score = result.get("score", 0.0)
            print(f"  {idx}. {title} (Similarity: {score:.2%})")
    
    # Print RAG context preview
    print("\n" + "="*80)
    print("RAG CONTEXT PREVIEW")
    print("="*80)
    if rag_results:
        print(f"Showing first result (of {len(rag_results)}):")
        first_result = rag_results[0]
        metadata = first_result.get("metadata", {})
        print(f"Title: {metadata.get('title', 'Unknown')}")
        print(f"Similarity Score: {first_result.get('score', 0):.2%}")
        print(f"Content Preview:\n{first_result['content'][:500]}...")
    else:
        print("No RAG results found. Check vector store and similarity threshold.")
    print("="*80)
    
    # Call LLM
    print("\nCalling LLM API with RAG-enhanced prompt...")
    llm_response = call_llm_api(prompt)
    
    if llm_response:
        print("\n" + "="*80)
        print("LLM ACTION PLAN")
        print("="*80)
        print(f"Threat Level: {llm_response.get('threat_level', 'Unknown')}")
        print(f"Priority: {llm_response.get('execution_priority', 'Unknown')}")
        print(f"\nPrimary Actions:")
        for i, action in enumerate(llm_response.get('primary_actions', [])[:5], 1):
            print(f"  {i}. {action}")
        
        print(f"\nSupporting Actions:")
        for i, action in enumerate(llm_response.get('supporting_actions', [])[:3], 1):
            print(f"  {i}. {action}")
        
        if llm_response.get('knowledge_sources_used'):
            print(f"\nKnowledge Sources Used:")
            for source in llm_response.get('knowledge_sources_used', [])[:3]:
                print(f"  - {source}")
        
        # Save result with enhanced metadata
        result = {
            "sample_id": sample.get("sample_id"),
            "prediction": {
                "predicted_label": sample.get("predicted_label"),
                "confidence": sample.get("confidence", 0.0)
            },
            "rag_info": {
                "documents_used": len(rag_results),
                "search_method": "vector_similarity",
                "top_results": [
                    {
                        "title": r.get("metadata", {}).get("title", "Unknown"),
                        "similarity_score": r.get("score", 0.0),
                        "source_file": r.get("metadata", {}).get("file", "unknown")
                    }
                    for r in rag_results[:3]
                ]
            },
            "llm_action_plan": llm_response
        }
        
        output_file = results_action_dir / "vector_rag_action_plan.json"
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(result, f, indent=2, ensure_ascii=False)
        
        print(f"\n✓ Saved to {output_file}")
    else:
        print("\n✗ Failed to get LLM response")